[Referenccce](https://medium.com/@harishk3493/the-ultimate-text-chunking-toolkit-15-methods-with-python-code-9ef9d8f6a898)

# 1. Fixed Chunking

In [1]:
def fixed_chunking(text, chunk_size=1000):
    """Split text into fixed-size chunks"""
    chunks = []
    for i in range(0, len(text), chunk_size):
        chunk = text[i:i + chunk_size]
        chunks.append(chunk)
    return chunks

# Usage
text = "Your long document text here..."
chunks = fixed_chunking(text, chunk_size=500)

# 2. Overlapping Chunking

In [2]:
def overlapping_chunking(text, chunk_size=1000, overlap=200):
    """Split text with overlapping windows"""
    chunks = []
    start = 0

    while start < len(text):
        end = start + chunk_size
        chunk = text[start:end]
        chunks.append(chunk)

        if end >= len(text):
            break

        start = end - overlap

    return chunks

# Usage
chunks = overlapping_chunking(text, chunk_size=800, overlap=150)

In [3]:
chunks

['Your long document text here...']

# 3. Semantic Chunking

In [5]:
import nltk
nltk.download('punkt_tab')
from nltk.tokenize import sent_tokenize

def semantic_chunking(text, max_chunk_size=1000):
    """Split text at sentence boundaries while respecting size limits"""
    sentences = sent_tokenize(text)
    chunks = []
    current_chunk = ""

    for sentence in sentences:
        if len(current_chunk + sentence) <= max_chunk_size:
            current_chunk += sentence + " "
        else:
            if current_chunk:
                chunks.append(current_chunk.strip())
            current_chunk = sentence + " "

    if current_chunk:
        chunks.append(current_chunk.strip())

    return chunks

# Usage
chunks = semantic_chunking(text, max_chunk_size=600)

[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.


# 4. Recursive Character Chunking

In [6]:
import re

def recursive_character_chunking(text, chunk_size=1000, separators=None):
    """Recursively split text using different separators"""
    if separators is None:
        separators = ["\n\n", "\n", ". ", " ", ""]

    def _split_text(text, separators, chunk_size):
        if len(text) <= chunk_size:
            return [text]

        for separator in separators:
            if separator in text:
                parts = text.split(separator)
                chunks = []
                current_chunk = ""

                for part in parts:
                    test_chunk = current_chunk + separator + part if current_chunk else part

                    if len(test_chunk) <= chunk_size:
                        current_chunk = test_chunk
                    else:
                        if current_chunk:
                            chunks.append(current_chunk)
                        current_chunk = part

                if current_chunk:
                    chunks.append(current_chunk)

                # Recursively split large chunks
                final_chunks = []
                for chunk in chunks:
                    if len(chunk) > chunk_size:
                        final_chunks.extend(_split_text(chunk, separators[1:], chunk_size))
                    else:
                        final_chunks.append(chunk)

                return final_chunks

        # If no separator works, split by characters
        return [text[i:i+chunk_size] for i in range(0, len(text), chunk_size)]

    return _split_text(text, separators, chunk_size)

# Usage
chunks = recursive_character_chunking(text, chunk_size=800)

# 5. Agentic Chunking

In [7]:
from transformers import pipeline
from nltk.tokenize import sent_tokenize

def agentic_chunking(text, model_name="facebook/bart-large-mnli", max_chunk_size=1000):
    """Use AI to determine optimal chunk boundaries"""
    classifier = pipeline("zero-shot-classification", model=model_name)
    sentences = sent_tokenize(text)

    chunks = []
    current_chunk = ""
    current_topic = None

    for sentence in sentences:
        # Classify sentence topic
        candidate_labels = ["introduction", "main_content", "conclusion", "transition"]
        result = classifier(sentence, candidate_labels)
        sentence_topic = result['labels'][0]

        # Check if we should start a new chunk
        if (current_topic and current_topic != sentence_topic) or \
           (len(current_chunk + sentence) > max_chunk_size):
            if current_chunk:
                chunks.append(current_chunk.strip())
            current_chunk = sentence + " "
            current_topic = sentence_topic
        else:
            current_chunk += sentence + " "
            current_topic = sentence_topic

    if current_chunk:
        chunks.append(current_chunk.strip())

    return chunks

# Usage
chunks = agentic_chunking(text, max_chunk_size=700)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/1.63G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/515 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

# 6. Advanced Semantic Chunking

In [8]:
import spacy

def advanced_semantic_chunking(text, max_chunk_size=1000, similarity_threshold=0.7):
    """Advanced chunking using multiple NLP features"""
    nlp = spacy.load("en_core_web_sm")
    doc = nlp(text)

    sentences = [sent.text for sent in doc.sents]
    chunks = []
    current_chunk = []
    current_entities = set()

    for i, sentence in enumerate(sentences):
        sent_doc = nlp(sentence)
        sent_entities = {ent.label_ for ent in sent_doc.ents}

        # Calculate entity overlap with current chunk
        if current_entities:
            overlap = len(current_entities.intersection(sent_entities)) / len(current_entities.union(sent_entities))
        else:
            overlap = 1.0

        chunk_text = " ".join(current_chunk + [sentence])

        # Start new chunk if low similarity or size exceeded
        if overlap < similarity_threshold or len(chunk_text) > max_chunk_size:
            if current_chunk:
                chunks.append(" ".join(current_chunk))
            current_chunk = [sentence]
            current_entities = sent_entities
        else:
            current_chunk.append(sentence)
            current_entities.update(sent_entities)

    if current_chunk:
        chunks.append(" ".join(current_chunk))

    return chunks

# Usage
chunks = advanced_semantic_chunking(text, max_chunk_size=800)

# 7. Context Enriched Chunking

In [9]:
import hashlib
from datetime import datetime
from nltk.tokenize import sent_tokenize

def context_enriched_chunking(text, chunk_size=1000, document_metadata=None):
    """Create chunks with rich context metadata"""
    sentences = sent_tokenize(text)
    chunks = []

    for i in range(0, len(sentences), 3):  # Group sentences
        sentence_group = sentences[i:i+3]
        chunk_text = " ".join(sentence_group)

        if len(chunk_text) > chunk_size:
            # Split large groups
            chunk_text = chunk_text[:chunk_size]

        # Create rich chunk object
        chunk = {
            'text': chunk_text,
            'chunk_id': hashlib.md5(chunk_text.encode()).hexdigest()[:8],
            'position': i // 3,
            'sentence_count': len(sentence_group),
            'char_count': len(chunk_text),
            'word_count': len(chunk_text.split()),
            'created_at': datetime.now().isoformat(),
            'source_document': document_metadata.get('filename') if document_metadata else 'unknown',
            'preceding_context': sentences[max(0, i-1)] if i > 0 else None,
            'following_context': sentences[min(len(sentences)-1, i+3)] if i+3 < len(sentences) else None
        }

        chunks.append(chunk)

    return chunks

# Usage
metadata = {'filename': 'document.pdf', 'author': 'John Doe'}
chunks = context_enriched_chunking(text, chunk_size=600, document_metadata=metadata)

# 8. Paragraph Chunking

In [10]:
import re
from nltk.tokenize import sent_tokenize

def paragraph_chunking(text, max_chunk_size=1000):
    """Split text at paragraph boundaries"""
    # Split by paragraph breaks (double newlines)
    paragraphs = re.split(r'\n\s*\n', text)

    chunks = []
    current_chunk = ""

    for paragraph in paragraphs:
        paragraph = paragraph.strip()
        if not paragraph:
            continue

        test_chunk = current_chunk + "\n\n" + paragraph if current_chunk else paragraph

        if len(test_chunk) <= max_chunk_size:
            current_chunk = test_chunk
        else:
            # If current chunk has content, save it
            if current_chunk:
                chunks.append(current_chunk)

            # If single paragraph is too large, split it further
            if len(paragraph) > max_chunk_size:
                # Split long paragraph at sentence boundaries
                sentences = sent_tokenize(paragraph)
                temp_chunk = ""
                for sentence in sentences:
                    if len(temp_chunk + sentence) <= max_chunk_size:
                        temp_chunk += sentence + " "
                    else:
                        if temp_chunk:
                            chunks.append(temp_chunk.strip())
                        temp_chunk = sentence + " "
                if temp_chunk:
                    chunks.append(temp_chunk.strip())
                current_chunk = ""
            else:
                current_chunk = paragraph

    if current_chunk:
        chunks.append(current_chunk)

    return chunks

# Usage
chunks = paragraph_chunking(text, max_chunk_size=800)

# 9. Recursive Sentence Chunking

In [11]:
def recursive_sentence_chunking(text, max_chunk_size=1000, min_chunk_size=100):
    """Recursively chunk text at sentence boundaries"""

    def split_sentences(text, max_size, min_size):
        if len(text) <= max_size:
            return [text]

        sentences = sent_tokenize(text)

        if len(sentences) == 1:
            # Single long sentence - split at clause boundaries
            clauses = re.split(r'[,;]', sentences[0])
            chunks = []
            current_chunk = ""

            for clause in clauses:
                test_chunk = current_chunk + clause
                if len(test_chunk) <= max_size:
                    current_chunk = test_chunk
                else:
                    if current_chunk and len(current_chunk) >= min_size:
                        chunks.append(current_chunk.strip())
                    current_chunk = clause

            if current_chunk:
                chunks.append(current_chunk.strip())

            return chunks

        # Multiple sentences - group them
        chunks = []
        current_chunk = ""

        for sentence in sentences:
            test_chunk = current_chunk + " " + sentence if current_chunk else sentence

            if len(test_chunk) <= max_size:
                current_chunk = test_chunk
            else:
                if current_chunk and len(current_chunk) >= min_size:
                    chunks.append(current_chunk.strip())

                # Recursively handle long sentences
                if len(sentence) > max_size:
                    chunks.extend(split_sentences(sentence, max_size, min_size))
                    current_chunk = ""
                else:
                    current_chunk = sentence

        if current_chunk and len(current_chunk) >= min_size:
            chunks.append(current_chunk.strip())

        return chunks

    return split_sentences(text, max_chunk_size, min_chunk_size)

# Usage
chunks = recursive_sentence_chunking(text, max_chunk_size=700, min_chunk_size=50)

# 10. Token Based Chunking

In [12]:
import tiktoken

def token_based_chunking(text, max_tokens=1000, model="gpt-3.5-turbo", overlap_tokens=50):
    """Split text based on token count for LLM processing"""
    # Initialize tokenizer for the specific model
    try:
        encoding = tiktoken.encoding_for_model(model)
    except:
        encoding = tiktoken.get_encoding("cl100k_base")  # Default encoding

    # Tokenize the entire text
    tokens = encoding.encode(text)

    chunks = []
    start = 0

    while start < len(tokens):
        end = start + max_tokens
        chunk_tokens = tokens[start:end]

        # Decode tokens back to text
        chunk_text = encoding.decode(chunk_tokens)
        chunks.append({
            'text': chunk_text,
            'token_count': len(chunk_tokens),
            'start_token': start,
            'end_token': min(end, len(tokens))
        })

        # Move start position (with overlap if specified)
        if end >= len(tokens):
            break
        start = end - overlap_tokens if overlap_tokens > 0 else end

    return chunks

# Alternative: Simple token counting without tiktoken
def simple_token_chunking(text, max_tokens=1000):
    """Simple approximation of token-based chunking"""
    # Rough approximation: 1 token ≈ 4 characters
    max_chars = max_tokens * 4
    return fixed_chunking(text, chunk_size=max_chars)

# Usage
chunks = token_based_chunking(text, max_tokens=500, model="gpt-4")

# 11. Document Structure-Aware Chunking

In [13]:
import re
from typing import List, Dict

def document_structure_chunking(text: str, max_chunk_size: int = 1000) -> List[Dict]:
    """Chunk text based on document structure markers"""

    # Define structure markers in order of priority
    structure_markers = [
        (r'^# .+', 'h1'),           # Markdown H1
        (r'^## .+', 'h2'),          # Markdown H2
        (r'^### .+', 'h3'),         # Markdown H3
        (r'^\d+\.\s+', 'numbered'),  # Numbered sections
        (r'^\* .+', 'bullet'),       # Bullet points
        (r'\n\n', 'paragraph')       # Paragraph breaks
    ]

    chunks = []
    lines = text.split('\n')
    current_chunk = []
    current_level = None

    for line in lines:
        line_type = None

        # Identify line type
        for pattern, level in structure_markers:
            if re.match(pattern, line):
                line_type = level
                break

        # Check if we should start a new chunk
        if line_type and line_type in ['h1', 'h2', 'h3', 'numbered']:
            if current_chunk and len('\n'.join(current_chunk)) > 50:
                chunks.append({
                    'text': '\n'.join(current_chunk).strip(),
                    'structure_level': current_level,
                    'char_count': len('\n'.join(current_chunk))
                })
            current_chunk = [line]
            current_level = line_type
        else:
            current_chunk.append(line)

            # Check size limit
            if len('\n'.join(current_chunk)) > max_chunk_size:
                if len(current_chunk) > 1:
                    # Save all but last line
                    chunks.append({
                        'text': '\n'.join(current_chunk[:-1]).strip(),
                        'structure_level': current_level,
                        'char_count': len('\n'.join(current_chunk[:-1]))
                    })
                    current_chunk = [current_chunk[-1]]

    # Add remaining chunk
    if current_chunk:
        chunks.append({
            'text': '\n'.join(current_chunk).strip(),
            'structure_level': current_level,
            'char_count': len('\n'.join(current_chunk))
        })

    return chunks

# Usage
chunks = document_structure_chunking(text, max_chunk_size=800)

# 12. Sliding Window Chunking

In [14]:
def sliding_window_chunking(text: str, window_size: int = 1000, step_size: int = 200) -> List[Dict]:
    """Create chunks using a sliding window approach"""

    chunks = []
    text_length = len(text)
    position = 0
    chunk_id = 0

    while position < text_length:
        # Extract window
        end_position = min(position + window_size, text_length)
        chunk_text = text[position:end_position]

        # Try to end at word boundary if not at end of text
        if end_position < text_length:
            last_space = chunk_text.rfind(' ')
            if last_space > window_size * 0.8:  # Only adjust if we don't lose too much
                chunk_text = chunk_text[:last_space]
                end_position = position + last_space

        chunks.append({
            'text': chunk_text,
            'chunk_id': chunk_id,
            'start_pos': position,
            'end_pos': end_position,
            'window_size': len(chunk_text),
            'overlap_with_previous': max(0, (position + window_size) - end_position) if chunk_id > 0 else 0
        })

        # Move window
        position += step_size
        chunk_id += 1

        # Break if we've covered the entire text
        if end_position >= text_length:
            break

    return chunks

# Usage
chunks = sliding_window_chunking(text, window_size=800, step_size=400)

# 13. Hierarchical Chunking

In [15]:
from typing import List, Dict, Any
from nltk.tokenize import sent_tokenize

def hierarchical_chunking(text: str, levels: List[int] = [2000, 1000, 500]) -> Dict[str, Any]:
    """Create hierarchical chunks at multiple levels"""

    def create_level_chunks(text: str, chunk_size: int, level_name: str) -> List[Dict]:
        sentences = sent_tokenize(text)
        chunks = []
        current_chunk = ""
        chunk_id = 0

        for sentence in sentences:
            test_chunk = current_chunk + " " + sentence if current_chunk else sentence

            if len(test_chunk) <= chunk_size:
                current_chunk = test_chunk
            else:
                if current_chunk:
                    chunks.append({
                        'text': current_chunk.strip(),
                        'level': level_name,
                        'chunk_id': f"{level_name}_{chunk_id}",
                        'char_count': len(current_chunk),
                        'sentence_count': len(sent_tokenize(current_chunk))
                    })
                    chunk_id += 1
                current_chunk = sentence

        if current_chunk:
            chunks.append({
                'text': current_chunk.strip(),
                'level': level_name,
                'chunk_id': f"{level_name}_{chunk_id}",
                'char_count': len(current_chunk),
                'sentence_count': len(sent_tokenize(current_chunk))
            })

        return chunks

    # Create hierarchy
    hierarchy = {
        'source_text': text,
        'levels': {},
        'relationships': []
    }

    # Generate chunks for each level
    level_names = ['coarse', 'medium', 'fine']
    for i, chunk_size in enumerate(levels):
        level_name = level_names[i] if i < len(level_names) else f'level_{i}'
        hierarchy['levels'][level_name] = create_level_chunks(text, chunk_size, level_name)

    # Create parent-child relationships
    for i in range(len(levels) - 1):
        parent_level = level_names[i]
        child_level = level_names[i + 1]

        for parent_chunk in hierarchy['levels'][parent_level]:
            parent_text = parent_chunk['text']
            children = []

            for child_chunk in hierarchy['levels'][child_level]:
                child_text = child_chunk['text']
                if child_text in parent_text:
                    children.append(child_chunk['chunk_id'])

            hierarchy['relationships'].append({
                'parent': parent_chunk['chunk_id'],
                'children': children,
                'relationship_type': f"{parent_level}_to_{child_level}"
            })

    return hierarchy

# Usage
hierarchy = hierarchical_chunking(text, levels=[1500, 800, 400])

# 14. Density-Based Chunking

In [16]:
import re
from nltk.tokenize import sent_tokenize

def density_based_chunking(text: str, target_density: float = 0.7, max_chunk_size: int = 1000) -> List[Dict]:
    """Chunk text based on information density"""

    def calculate_density(text_segment: str) -> float:
        """Calculate information density of a text segment"""
        words = re.findall(r'\b\w+\b', text_segment.lower())
        if not words:
            return 0.0

        # Metrics for density calculation
        unique_words = len(set(words))
        total_words = len(words)
        avg_word_length = sum(len(word) for word in words) / total_words

        # Simple density formula (can be enhanced)
        lexical_diversity = unique_words / total_words
        density = (lexical_diversity * avg_word_length) / 10  # Normalized

        return min(density, 1.0)

    sentences = sent_tokenize(text)
    chunks = []
    current_chunk = ""
    current_density = 0.0

    for sentence in sentences:
        test_chunk = current_chunk + " " + sentence if current_chunk else sentence
        test_density = calculate_density(test_chunk)

        # Check if adding sentence maintains target density
        density_diff = abs(test_density - target_density)
        current_diff = abs(current_density - target_density)

        if (len(test_chunk) <= max_chunk_size and
            (density_diff <= current_diff or len(current_chunk) < 100)):
            current_chunk = test_chunk
            current_density = test_density
        else:
            if current_chunk:
                chunks.append({
                    'text': current_chunk.strip(),
                    'density': current_density,
                    'char_count': len(current_chunk),
                    'density_score': f"{current_density:.3f}"
                })
            current_chunk = sentence
            current_density = calculate_density(sentence)

    if current_chunk:
        chunks.append({
            'text': current_chunk.strip(),
            'density': current_density,
            'char_count': len(current_chunk),
            'density_score': f"{current_density:.3f}"
        })

    return chunks

# Usage
chunks = density_based_chunking(text, target_density=0.6, max_chunk_size=800)

# 15. Adaptive Threshold Chunking

In [17]:
import statistics
from nltk.tokenize import sent_tokenize

def adaptive_threshold_chunking(text: str, base_chunk_size: int = 1000, adaptation_factor: float = 0.3) -> List[Dict]:
    """Dynamically adapt chunk size based on content characteristics"""

    def analyze_text_characteristics(text_segment: str) -> Dict[str, float]:
        """Analyze characteristics of text segment"""
        sentences = sent_tokenize(text_segment)
        words = text_segment.split()

        return {
            'avg_sentence_length': statistics.mean([len(s.split()) for s in sentences]) if sentences else 0,
            'sentence_count': len(sentences),
            'avg_word_length': statistics.mean([len(w) for w in words]) if words else 0,
            'punctuation_density': len(re.findall(r'[.!?;:]', text_segment)) / len(text_segment) if text_segment else 0
        }

    def calculate_adaptive_threshold(characteristics: Dict[str, float], base_size: int) -> int:
        """Calculate adaptive threshold based on text characteristics"""
        # Adjust based on sentence complexity
        sentence_factor = min(characteristics['avg_sentence_length'] / 15, 2.0)  # Normalize around 15 words
        word_factor = min(characteristics['avg_word_length'] / 5, 1.5)  # Normalize around 5 chars
        punctuation_factor = min(characteristics['punctuation_density'] * 100, 1.3)  # Dense punctuation

        # Calculate adjustment
        complexity_score = (sentence_factor + word_factor + punctuation_factor) / 3
        adjustment = 1 + (complexity_score - 1) * adaptation_factor

        return int(base_size * adjustment)

    # Initial analysis of entire text
    global_characteristics = analyze_text_characteristics(text)
    sentences = sent_tokenize(text)

    chunks = []
    current_chunk = ""
    window_start = 0

    while window_start < len(sentences):
        # Analyze current window
        window_end = min(window_start + 5, len(sentences))  # Look ahead 5 sentences
        window_text = " ".join(sentences[window_start:window_end])
        local_characteristics = analyze_text_characteristics(window_text)

        # Calculate adaptive threshold
        adaptive_size = calculate_adaptive_threshold(local_characteristics, base_chunk_size)

        # Build chunk with adaptive threshold
        for i in range(window_start, len(sentences)):
            test_chunk = current_chunk + " " + sentences[i] if current_chunk else sentences[i]

            if len(test_chunk) <= adaptive_size:
                current_chunk = test_chunk
            else:
                if current_chunk:
                    chunk_characteristics = analyze_text_characteristics(current_chunk)
                    chunks.append({
                        'text': current_chunk.strip(),
                        'adaptive_size_used': adaptive_size,
                        'actual_size': len(current_chunk),
                        'complexity_score': (local_characteristics['avg_sentence_length'] +
                                           local_characteristics['avg_word_length'] +
                                           local_characteristics['punctuation_density']) / 3,
                        'characteristics': chunk_characteristics
                    })
                current_chunk = sentences[i]
                window_start = i
                break
        else:
            break

    if current_chunk:
        chunks.append({
            'text': current_chunk.strip(),
            'adaptive_size_used': adaptive_size,
            'actual_size': len(current_chunk),
            'complexity_score': 0,
            'characteristics': analyze_text_characteristics(current_chunk)
        })

    return chunks

# Usage
chunks = adaptive_threshold_chunking(text, base_chunk_size=800, adaptation_factor=0.25)